# Reading csv file


### init


In [0]:
import logging
from pyspark.sql.functions import lit, current_date
from datetime import datetime

In [0]:
def log(step, message):
    print(f"[{step}] {message}")

In [0]:
log("Bronze", "Reading data")

df = (spark.read.option("header","true").option("inferSchema","true").csv("/Volumes/workspace/bronze/source_systems/retail.csv"))

log("Bronze", "Rows loaded succesfully")

In [0]:
df = df.toDF(*[col.replace(' ', '_').replace('-', '_') for col in df.columns])

### Adding metadata columns (file_path, execution_datetime) 


In [0]:
log("Bronze", "Adding metadata: execution_datetime, source_file")
execution_time = datetime.now()

df = (
    df
    .withColumn("execution_datetime", lit(execution_time))
    .withColumn("source_file", lit("retail.csv"))
)

In [0]:
df.display()

# Write to Bronze layer


In [0]:
log("Bronze", "Writing table bronze.sales_details")
try:
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze.sales_details")
    log("Silver", "Write Succesful")
except Exception as e:
    log("Bronze", f"Write failed: {str(e)}")
    raise
